In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

CANDIDATES_PATH = Path(
    '/home/msp25gd/ResearchProjectMSc/ResolutionHandling/QuickSearch_V2/'
    'candidates_-3.5sig_1.5cut_2width_V2_RES.npy'
)
DATASET_ROOT = Path(
    '/home/msp25gd/ResearchProjectMSc/ResolutionHandling/processed_candidates'
)

candidates = np.load(CANDIDATES_PATH, allow_pickle=True).astype(str)
if len(candidates) != 492:
    raise ValueError(f'Expected 492 QuickSearch candidates, found {len(candidates)}')

# QuickSearch candidate names correspond to folders in the processed dataset.
folder_lookup = {
    path.name.casefold(): path
    for path in DATASET_ROOT.iterdir()
    if path.is_dir()
}

records = []
missing_candidates = []
for candidate in candidates:
    group_path = folder_lookup.get(candidate.casefold())
    if group_path is None:
        missing_candidates.append(candidate)
        continue

    spectra_path = group_path / 'spec' / 'sK.npy'
    if not spectra_path.exists():
        raise FileNotFoundError(f'Missing spectra file: {spectra_path}')

    records.append({
        'stellar_group': candidate,
        'n_spectra': np.load(spectra_path, mmap_mode='r').shape[0],
    })

if missing_candidates:
    raise ValueError(
        f'Could not match {len(missing_candidates)} candidates to dataset folders: '
        f'{missing_candidates[:10]}'
    )

candidate_groups = pd.DataFrame(records)
labels = ['3-9', '10-19', '20-29', '30+']
candidate_groups['spectra_bin'] = pd.cut(
    candidate_groups['n_spectra'],
    bins=[3, 10, 20, 30, np.inf],
    labels=labels,
    right=False,
)

if candidate_groups['spectra_bin'].isna().any():
    invalid = candidate_groups.loc[candidate_groups['spectra_bin'].isna()]
    raise ValueError(f'Candidates with fewer than 3 spectra found:\n{invalid}')

# Four arrays containing the candidate names in each requested range.
grouped_candidates = {
    label: candidate_groups.loc[
        candidate_groups['spectra_bin'] == label, 'stellar_group'
    ].to_numpy()
    for label in labels
}

summary = (
    candidate_groups['spectra_bin']
    .value_counts(sort=False)
    .rename_axis('spectra_per_group')
    .reset_index(name='stellar_groups')
)

print(f'QuickSearch candidates divided: {len(candidate_groups)}')
display(summary)
